# Problem 86

A spider, S, sits in one corner of a cuboid room, measuring $6$ by $5$ by $3$, and a fly, F, sits in the opposite corner. By travelling on the surfaces of the room the shortest "straight line" distance from S to F is $10$ and the path is shown on the diagram.
<img src="resources/images/0086.png?1678992052" class="dark_img" alt=""><br>
However, there are up to three "shortest" path candidates for any given cuboid and the shortest route doesn't always have integer length.
It can be shown that there are exactly $2060$ distinct cuboids, ignoring rotations, with integer dimensions, up to a maximum size of $M$ by $M$ by $M$, for which the shortest route has integer length when $M = 100$. This is the least value of $M$ for which the number of solutions first exceeds two thousand; the number of solutions when $M = 99$ is $1975$.
Find the least value of $M$ such that the number of solutions first exceeds one million.


In [49]:
# This problem turns into pushing the back wall of the cube flat and looking at right triangles,
# segmenting right leg of pythagorean triples.

import math
from math import gcd, isqrt

def pythagorean_triples(max_hypotenuse):
    triples = []
    for n in range(1, isqrt(max_hypotenuse // 2) + 1):
        for m in range(n + 1, isqrt(max_hypotenuse) + 1, 2):
            if gcd(m, n) != 1:
                continue
            a = m * m - n * n
            b = 2 * m * n
            c = m * m + n * n
            if c > max_hypotenuse:
                break
            k = 1
            while k * c <= max_hypotenuse:
                triples.append(sorted((k * a, k * b, k * c))) # sort for filtering later
                k += 1
    return sorted(triples, key=lambda t: (t[2], t[0])) # order by hypotenuse, then by shorter leg

over_M = 2000 # overshoot and backtrack to find first value over instead of regenerating

triples = pythagorean_triples(math.ceil(math.sqrt(over_M**2 + (2*over_M)**2))) # max hypotenuse for MxMxM
triples = [triple for triple in triples if triple[0] <= over_M and triple[1] <= 2*over_M] # keep triangles that satisfy size

cuboids_at_M = [0] * (over_M + 1)
for triple in triples:
    a, b, c = triple # a < b < c

    # case 1 b is folded, a is not folded (only to half to AVOID rotations)
    if b <= 2*a: 
        # the unfolded side MUST remain the largest side of the cuboid
        # the math:
        # the possible unfoldings pair (a,b), (a,c), (b,c), distances (a+b)^2 + c^2, (a+c)^2 + b^2, (b+c)^2 + a^2
        # all have a^2 + b^2 + c^2 only difference is ab vs. ac vs. bc. the min takes the two smallest sides
        # so the unfolded side must be the largest side or the min distance won't be integer
        cuboids_at_M[a] += a - (b - 1) // 2
    
    # case 2 a is folded, b is not
    if b <= over_M: # unfolded side being longest is automatically satisfied
        cuboids_at_M[b] += a // 2

route_sum = 0
for i in range(1, over_M + 1):
    route_sum += cuboids_at_M[i]
    if route_sum > 1000000:
        print(i)
        break

1818


# Problem 87

The smallest number expressible as the sum of a prime square, prime cube, and prime fourth power is $28$. In fact, there are exactly four numbers below fifty that can be expressed in such a way:
$$\begin{align*}
28 &= 2^2 + 2^3 + 2^4\\
33 &= 3^2 + 2^3 + 2^4\\
49 &= 5^2 + 2^3 + 2^4\\
47 &= 2^2 + 3^3 + 2^4
\end{align*}$$
How many numbers below fifty million can be expressed as the sum of a prime square, prime cube, and prime fourth power?


In [9]:
# max ^2 is 7071, max ^3 is 368, max ^4 is 84
from sympy import primerange

squares = list(primerange(1, 7071))
cubes = list(primerange(1, 368))
fourths = list(primerange(1, 84))

sums = set()
for s in squares:
    for c in cubes:
        for f in fourths:
            if s**2 + c**3 + f**4 >= 50000000:
                break
            sums.add(s**2 + c**3 + f**4)
print(len(sums))

1097343


# Problem 88

A natural number, $N$, that can be written as the sum and product of a given set of at least two natural numbers, $\{a_1, a_2, \dots, a_k\}$ is called a product-sum number: $N = a_1 + a_2 + \cdots + a_k = a_1 \times a_2 \times \cdots \times a_k$.
For example, $6 = 1 + 2 + 3 = 1 \times 2 \times 3$.
For a given set of size, $k$, we shall call the smallest $N$ with this property a minimal product-sum number. The minimal product-sum numbers for sets of size, $k = 2, 3, 4, 5$, and $6$ are as follows.
\begin{align*}
&k=2: 4 = 2 \times 2 = 2 + 2 \\
&k=3: 6 = 1 \times 2 \times 3 = 1 + 2 + 3 \\
&k=4: 8 = 1 \times 1 \times 2 \times 4 = 1 + 1 + 2 + 4 \\
&k=5: 8 = 1 \times 1 \times 2 \times 2 \times 2 = 1 + 1 + 2 + 2 + 2 \\
&k=6: 12 = 1 \times 1 \times 1 \times 1 \times 2 \times 6 = 1 + 1 + 1 + 1 + 2 + 6
\end{align*}
Hence for $2 \le k \le 6$, the sum of all the minimal product-sum numbers is $4+6+8+12 = 30$; note that $8$ is only counted once in the sum.
In fact, as the complete set of minimal product-sum numbers for $2 \le k \le 12$ is $\{4, 6, 8, 12, 15, 16\}$, the sum is $61$.
What is the sum of all the minimal product-sum numbers for $2 \le k \le 12000$?


In [ ]:
def generate_all(k_max):
    best = {} # key k -> min product-sum number
    P_limit = 2 * k_max

    def recurse(P, S, j, min_factor): # product, sum, # of factors, min factor to use
        k = j + (P - S) # number of padding 1's needed to make the sum equal to the product
        if 2 <= k <= k_max:
            if k not in best or P < best[k]: # new or better solution
                best[k] = P
        for f in range(min_factor, P_limit // max(P, 1) + 1): # possibile remaining factors to add to set
            new_P = P * f
            if new_P > P_limit:
                break
            recurse(new_P, S + f, j + 1, f)

    recurse(1, 0, 0, 2)
    return best


best_n = generate_all(12000)
print(sum(set(best_n.values())))

7587457


# Problem 89
For a number written in Roman numerals to be considered valid there are basic rules which must be followed. Even though the rules allow some numbers to be expressed in more than one way there is always a "best" way of writing a particular number.
For example, it would appear that there are at least six ways of writing the number sixteen:
<p class="margin_left monospace">IIIIIIIIIIIIIIII<br>
VIIIIIIIIIII<br>
VVIIIIII<br>
XIIIIII<br>
VVVI<br>
XVI</p>
However, according to the rules only <span class="monospace">XIIIIII</span> and <span class="monospace">XVI</span> are valid, and the last example is considered to be the most efficient, as it uses the least number of numerals.
The 11K text file, <a href="resources/documents/0089_roman.txt">roman.txt</a> (right click and 'Save Link/Target As...'), contains one thousand numbers written in valid, but not necessarily minimal, Roman numerals; see <a href="about=roman_numerals">About... Roman Numerals</a> for the definitive rules for this problem.
Find the number of characters saved by writing each of these in their minimal form.
<p class="smaller">Note: You can assume that all the Roman numerals in the file contain no more than four consecutive identical units.</p>


In [ ]:
values = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}

with open('0089_roman.txt') as f:
    lines = f.read().splitlines()
numerals = [line.strip() for line in lines]

# read numeral to integer
def roman_to_int(numeral):
    total = 0
    for i, num in enumerate(numeral):
        if i - 1 >= 0 and values[numeral[i - 1]] < values[num]:
            continue  # skip if previous numeral is smaller (subtractive notation)
        # if next numeral is larger, subtract current numeral
        if i + 1 < len(numeral) and values[numeral[i + 1]] > values[numeral[i]]:
            if values[numeral[i + 1]] > 10 * values[numeral[i]]:
                raise ValueError(f"Invalid numeral: {numeral}")
            total += values[numeral[i + 1]] - values[numeral[i]]
        else:
            total += values[numeral[i]]
        
    return total

def int_to_roman(num): # doesn't actually return the numeral, just the length of it
    total = 0 # length of numeral, but not numeral itself
    digits = str(num)[::-1] # ones, tens, hundreds, thousands, etc.
    for i, digit in enumerate(digits):
        digit = int(digit)
        if i >= 3:
            total += digit # one M for each thousand
        elif digit % 5 == 0:
            total += digit // 5
        elif digit % 5 == 1:
            total += digit // 5 + 1 # 1 vs 6
        elif digit % 5 == 2:
            total += digit // 5 + 2 # 2 vs 7
        elif digit % 5 == 3:
            total += digit // 5 + 3 # 3 vs 8
        elif digit % 5 == 4:
            total += 2 # 4 is IV, 9 is IX, both are two characters
    return total

savings = 0
for numeral in numerals:
    num = roman_to_int(numeral)
    savings += len(numeral) - int_to_roman(num)
    
print(savings)

743


# Problem 90

Each of the six faces on a cube has a different digit ($0$ to $9$) written on it; the same is done to a second cube. By placing the two cubes side-by-side in different positions we can form a variety of $2$-digit numbers.

For example, the square number $64$ could be formed:

<div class="center">
<img src="resources/images/0090.png?1678992052" class="dark_img" alt=""><br></div>

In fact, by carefully choosing the digits on both cubes it is possible to display all of the square numbers below one-hundred: $01$, $04$, $09$, $16$, $25$, $36$, $49$, $64$, and $81$.

For example, one way this can be achieved is by placing $\{0, 5, 6, 7, 8, 9\}$ on one cube and $\{1, 2, 3, 4, 8, 9\}$ on the other cube.

However, for this problem we shall allow the $6$ or $9$ to be turned upside-down so that an arrangement like $\{0, 5, 6, 7, 8, 9\}$ and $\{1, 2, 3, 4, 6, 7\}$ allows for all nine square numbers to be displayed; otherwise it would be impossible to obtain $09$.

In determining a distinct arrangement we are interested in the digits on each cube, not the order.

$\{1, 2, 3, 4, 5, 6\}$ is equivalent to $\{3, 6, 4, 1, 2, 5\}$ \
$\{1, 2, 3, 4, 5, 6\}$ is distinct from $\{1, 2, 3, 4, 5, 9\}$

But because we are allowing $6$ and $9$ to be reversed, the two distinct sets in the last example both represent the extended set $\{1, 2, 3, 4, 5, 6, 9\}$ for the purpose of forming $2$-digit numbers.

How many distinct arrangements of the two cubes allow for all of the square numbers to be displayed?